In [ ]:
import itertools

import numpy as np
import pandas as pd
import yfinance as yf
import random
import matplotlib.pyplot as plt

import torch
import os
from stable_baselines3.common.utils import set_random_seed

from finrl import config_tickers
from finrl.config import INDICATORS, TRAIN_START_DATE, TRAIN_END_DATE, TRADE_START_DATE, TRADE_END_DATE
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from pypfopt.efficient_frontier import EfficientFrontier

In [ ]:
tickers = ["AAPL", "MSFT", "JPM", "XOM", "KO"]

train_start = "2018-01-01"
train_end   = "2023-01-01"  
test_start  = "2023-01-01"
test_end    = "2025-01-01"  

In [ ]:
train_raw = yf.download(
    tickers=tickers,
    start=train_start,
    end=train_end,
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    progress=False
)

trade_raw = yf.download(
    tickers=tickers,
    start=test_start,
    end=test_end,
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    progress=False)

In [ ]:
train_raw.head

In [ ]:
trade_raw.head

In [ ]:
def process_yf_to_finrl(df, tickers):
    temp_df = df.stack(level=0).reset_index()
    temp_df.columns = ['date', 'tic', 'open', 'high', 'low', 'close', 'volume']

    temp_df['date'] = temp_df['date'].dt.strftime('%Y-%m-%d')
    return temp_df


train_processed_raw = process_yf_to_finrl(train_raw, tickers)
trade_processed_raw = process_yf_to_finrl(trade_raw, tickers)

In [ ]:
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,  
    use_vix=False,                  
    use_turbulence=False,            
    user_defined_feature=False,
)


train_fe = fe.preprocess_data(train_processed_raw)
trade_fe = fe.preprocess_data(trade_processed_raw)

In [ ]:


def clean_data(df):
    list_ticker = df["tic"].unique().tolist()
    list_date = list(pd.date_range(df["date"].min(), df["date"].max()).astype(str))
    
    combination = list(itertools.product(list_date, list_ticker))
    processed_full = pd.DataFrame(combination, columns=["date", "tic"])
    

    processed_full = processed_full.merge(df, on=["date", "tic"], how="left")
    processed_full = processed_full[processed_full["date"].isin(df["date"])]
    processed_full = processed_full.sort_values(["date", "tic"])
    processed_full = processed_full.fillna(0)  #
    return processed_full

train_final = clean_data(train_fe)
trade_final = clean_data(trade_fe)

In [ ]:
train_save_path = "/Users/xuduo/Desktop/train_fe_with_indicators.csv"
trade_save_path = "/Users/xuduo/Desktop/trade_fe_with_indicators.csv"
train_final.to_csv(train_save_path, index=False)
trade_final.to_csv(trade_save_path, index=False)

In [ ]:
SEED = 123

def set_global_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_random_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.set_num_threads(1)

set_global_seed(SEED)

In [ ]:
train_final = pd.read_csv(train_save_path)
trade_final = pd.read_csv(trade_save_path)

train_final["date"] = pd.to_datetime(train_final["date"]).dt.strftime("%Y-%m-%d")
trade_final["date"] = pd.to_datetime(trade_final["date"]).dt.strftime("%Y-%m-%d")

train_final = train_final.sort_values(["date", "tic"]).reset_index(drop=True)
trade_final = trade_final.sort_values(["date", "tic"]).reset_index(drop=True)

train_final = train_final.fillna(0)
trade_final = trade_final.fillna(0)

train_final.index = train_final["date"].factorize()[0]
trade_final.index = trade_final["date"].factorize()[0]

In [ ]:
stock_dimension = len(train_final["tic"].unique())
state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": [0] * stock_dimension,
    "buy_cost_pct": [0.001] * stock_dimension,
    "sell_cost_pct": [0.001] * stock_dimension,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

e_train_gym = StockTradingEnv(df=train_final, **env_kwargs)
env_train, _ = e_train_gym.get_sb_env()


In [ ]:
agent = DRLAgent(env=env_train)
model_a2c = agent.get_model("a2c")
trained_a2c = agent.train_model(
    model=model_a2c,
    tb_log_name="a2c",
    total_timesteps=10000
)

In [ ]:
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.01,
    "learning_rate": 0.00025,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo", model_kwargs=PPO_PARAMS)
trained_ppo = agent.train_model(
    model=model_ppo,
    tb_log_name="ppo",
    total_timesteps=10000
)

In [ ]:
e_trade_gym_a2c = StockTradingEnv(df=trade_final, **env_kwargs)
e_trade_gym_ppo = StockTradingEnv(df=trade_final, **env_kwargs)

df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
    model=trained_a2c,
    environment=e_trade_gym_a2c
)

df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
    model=trained_ppo,
    environment=e_trade_gym_ppo
)

In [ ]:
%matplotlib inline

result_df = df_account_value_ppo[["date", "account_value"]].rename(columns={"account_value": "PPO"})
plt.figure(figsize=(15, 6))
plt.plot(result_df["date"], result_df["PPO"], label="RL Agent: PPO", linewidth=2, color="green")
plt.title("Portfolio Value Over Time For PPO", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Portfolio Value ($)", fontsize=12)
plt.ylim(0.7e6, 1.9e6)
plt.legend(fontsize=12, loc="upper left")
plt.grid(True, alpha=0.5)

xticks_interval = max(1, len(result_df) // 10)
plt.xticks(result_df["date"][::xticks_interval], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

result_df = df_account_value_a2c[["date", "account_value"]].rename(columns={"account_value": "A2C"})
plt.figure(figsize=(15, 6))
plt.plot(result_df["date"], result_df["A2C"], label="RL Agent: A2C", linewidth=2, color="blue")
plt.title("Portfolio Value Over Time For A2C", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Portfolio Value ($)", fontsize=12)
plt.ylim(0.7e6, 2e6)
plt.legend(fontsize=12, loc="upper left")
plt.grid(True, alpha=0.5)

xticks_interval = max(1, len(result_df) // 10)
plt.xticks(result_df["date"][::xticks_interval], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def process_df_for_mvo(df):
    return df.pivot(index="date", columns="tic", values="close")

StockData = process_df_for_mvo(train_final)
TradeData = process_df_for_mvo(trade_final)

returns = StockData.pct_change().dropna()
meanReturns = returns.mean()
covReturns = returns.cov()

ef = EfficientFrontier(meanReturns, covReturns, weight_bounds=(0, 1))
raw_weights = ef.max_sharpe()
cleaned_weights = ef.clean_weights()

mvo_weights = np.array([cleaned_weights[tic] for tic in TradeData.columns])
print(cleaned_weights)

initial_amount = 1000000
mvo_returns = (TradeData.pct_change().dropna() * mvo_weights).sum(axis=1)
mvo_portfolio_value = initial_amount * (1 + mvo_returns).cumprod()

mvo_result = pd.DataFrame({
    "date": mvo_portfolio_value.index,
    "MVO": mvo_portfolio_value.values
})
initial_row = pd.DataFrame({"date": [TradeData.index[0]], "MVO": [initial_amount]})
mvo_result = pd.concat([initial_row, mvo_result], ignore_index=True)

In [ ]:
%matplotlib inline

plt.figure(figsize=(15, 6))
plt.plot(
    mvo_result["date"],
    mvo_result["MVO"],
    label="Baseline: MVO",
    linewidth=2,
    color="red"
)

plt.title("Portfolio Value Over Time For MVO", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Portfolio Value ($)", fontsize=12)
plt.legend(fontsize=12, loc="upper left")
plt.grid(True, alpha=0.5)

xticks_interval = max(1, len(mvo_result) // 10)
plt.xticks(mvo_result["date"][::xticks_interval], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

result_df = df_account_value_a2c[["date", "account_value"]].rename(columns={"account_value": "A2C"})
result_df = result_df.merge(df_account_value_ppo[["date", "account_value"]].rename(columns={"account_value": "PPO"}), on="date")
result_df = result_df.merge(mvo_result, on="date")

plt.figure(figsize=(15, 6))
plt.plot(result_df["date"], result_df["A2C"], label="RL Agent: A2C", linewidth=2, color="blue")
plt.plot(result_df["date"], result_df["PPO"], label="RL Agent: PPO", linewidth=2, color="green")
plt.plot(result_df["date"], result_df["MVO"], label="Baseline: MVO", linewidth=2, linestyle='--', color="red")

plt.title("Portfolio Value Over Time: Reinforcement Learning vs MVO", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Portfolio Value ($)", fontsize=12)
plt.legend(fontsize=12, loc="upper left")
plt.grid(True, alpha=0.5)

xticks_interval = max(1, len(result_df) // 10)
plt.xticks(result_df["date"][::xticks_interval], rotation=45)

plt.tight_layout()
plt.show()